# EMG Gesture Classification Pipeline
**Sentry Robotics — Mr. Handy Project**

This notebook implements a full end-to-end pipeline for classifying hand gestures from 3-channel forearm EMG data:

1. **Data Loading & Exploration** — Load all 20 CSVs, inspect distributions, check class balance
2. **Preprocessing** — Bandpass filter, notch filter, normalization
3. **Windowing** — Sliding windows with overlap to create fixed-length samples
4. **Feature Engineering** — Time-domain + frequency-domain features per channel
5. **Model Training** — Two parallel approaches:
   - `CNN-1D` operating directly on raw windowed signals (end-to-end)
   - `MLP` operating on handcrafted features (interpretable baseline)
6. **Evaluation** — Accuracy, confusion matrix, per-class F1, training curves
7. **Export** — Save best model for deployment on Raspberry Pi

---
**CSV format assumed:** `ch1, ch2, ch3, label` — one row per time sample at **1000 Hz**

## 0. Install & Import Dependencies

In [ ]:
# Uncomment to install if running fresh
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip install scikit-learn pandas numpy matplotlib seaborn scipy tqdm

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis
from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 1. Configuration

In [ ]:
# ─── USER CONFIGURATION ─────────────────────────────────────────────────────

# Path to directory containing your 20 CSV files
# Can also be a list of explicit paths: DATA_PATHS = ['path/to/file1.csv', ...]
DATA_DIR = './data'           # <-- CHANGE THIS to your data folder
CSV_PATTERN = '*.csv'         # glob pattern to match your CSV files

# CSV column names (adjust if your headers differ)
CH1_COL    = 'ch1'           # Channel 1 raw EMG
CH2_COL    = 'ch2'           # Channel 2 raw EMG
CH3_COL    = 'ch3'           # Channel 3 raw EMG
LABEL_COL  = 'label'         # Gesture label column

# If your CSV has no header row, set HEADER=None and define column names:
HEADER     = 0               # 0 = first row is header; None = no header
# COL_NAMES = ['ch1','ch2','ch3','label']  # uncomment if HEADER=None

# Signal parameters
FS          = 1000           # Sampling rate in Hz
N_CHANNELS  = 3

# Windowing parameters
WINDOW_MS   = 200            # Window length in milliseconds → 200 samples @ 1kHz
OVERLAP_PCT = 0.5            # 50% overlap between consecutive windows
WINDOW_LEN  = int(FS * WINDOW_MS / 1000)   # samples per window
STEP_LEN    = int(WINDOW_LEN * (1 - OVERLAP_PCT))

# Filtering
BANDPASS_LOW  = 20           # Hz  — removes motion artifact
BANDPASS_HIGH = 450          # Hz  — anti-alias (below Nyquist of 500 Hz)
NOTCH_FREQ    = 60           # Hz  — power line noise (use 50 if in Europe)
FILTER_ORDER  = 4

# Training parameters
BATCH_SIZE    = 64
N_EPOCHS      = 60
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15

# Label majority vote: fraction of window that must be one class to keep window
LABEL_PURITY_THRESHOLD = 0.80

print(f'Window: {WINDOW_LEN} samples ({WINDOW_MS} ms)')
print(f'Step:   {STEP_LEN} samples ({STEP_LEN} ms)')
print(f'Bandpass: {BANDPASS_LOW}–{BANDPASS_HIGH} Hz | Notch: {NOTCH_FREQ} Hz')

## 2. Data Loading & Exploration

In [ ]:
def load_all_csvs(data_dir, pattern, header=0):
    """Load all matching CSVs from a directory into a single DataFrame.
    
    Adds a 'source_file' column so you can trace each row back to its file.
    """
    paths = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not paths:
        raise FileNotFoundError(
            f"No CSV files found at '{data_dir}/{pattern}'.\n"
            f"Please update DATA_DIR in the configuration cell."
        )
    
    dfs = []
    for path in paths:
        try:
            if header is None:
                df = pd.read_csv(path, header=None)
                df.columns = [CH1_COL, CH2_COL, CH3_COL, LABEL_COL]
            else:
                df = pd.read_csv(path, header=header)
            df['source_file'] = os.path.basename(path)
            dfs.append(df)
            print(f'  Loaded: {os.path.basename(path):40s} | {len(df):>7,} rows')
        except Exception as e:
            print(f'  FAILED: {os.path.basename(path)} — {e}')
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal rows loaded: {len(combined):,}')
    return combined, paths


raw_df, csv_paths = load_all_csvs(DATA_DIR, CSV_PATTERN, header=HEADER)

In [ ]:
# ─── Basic inspection ────────────────────────────────────────────────────────
print('Shape:', raw_df.shape)
print('\nDtypes:')
print(raw_df.dtypes)
print('\nFirst 5 rows:')
raw_df.head()

In [ ]:
# ─── Validate and clean ───────────────────────────────────────────────────────

# Ensure signal columns are numeric
signal_cols = [CH1_COL, CH2_COL, CH3_COL]
raw_df[signal_cols] = raw_df[signal_cols].apply(pd.to_numeric, errors='coerce')

# Check for NaNs
nan_counts = raw_df[signal_cols + [LABEL_COL]].isnull().sum()
print('NaN counts per column:')
print(nan_counts)

# Drop rows with NaNs in any signal or label column
before = len(raw_df)
raw_df.dropna(subset=signal_cols + [LABEL_COL], inplace=True)
print(f'\nDropped {before - len(raw_df)} rows with NaN values.')

# Stringify labels for safety
raw_df[LABEL_COL] = raw_df[LABEL_COL].astype(str).str.strip()

print(f'\nUnique gesture labels ({raw_df[LABEL_COL].nunique()} total):')
print(sorted(raw_df[LABEL_COL].unique()))

In [ ]:
# ─── Class distribution ───────────────────────────────────────────────────────
label_counts = raw_df[LABEL_COL].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Sample Count per Gesture Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gesture'); axes[0].set_ylabel('# Samples (rows)')
axes[0].tick_params(axis='x', rotation=45)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Pie chart
axes[1].pie(label_counts, labels=label_counts.index, autopct='%1.1f%%',
            startangle=90, pctdistance=0.8)
axes[1].set_title('Class Balance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(label_counts)

In [ ]:
# ─── Raw signal preview (first 2 seconds from each channel) ──────────────────
PREVIEW_SAMPLES = 2000  # 2 seconds @ 1kHz
preview = raw_df.head(PREVIEW_SAMPLES)

fig, axes = plt.subplots(3, 1, figsize=(15, 7), sharex=True)
t = np.arange(len(preview)) / FS

for i, col in enumerate(signal_cols):
    axes[i].plot(t, preview[col].values, linewidth=0.6, color=f'C{i}')
    axes[i].set_ylabel(f'{col} (ADC)', fontsize=10)
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Raw EMG Signal — First 2 Seconds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('raw_signal_preview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Signal Preprocessing

In [ ]:
def build_filters(fs, bp_low, bp_high, notch_freq, order=4):
    """Pre-compute Butterworth bandpass + notch filter coefficients."""
    nyq = fs / 2.0

    # Bandpass (20–450 Hz): remove motion artifact and alias noise
    bp_b, bp_a = scipy_signal.butter(
        order, [bp_low / nyq, bp_high / nyq], btype='bandpass'
    )

    # Notch at power-line frequency (60 Hz USA / 50 Hz EU)
    notch_b, notch_a = scipy_signal.iirnotch(notch_freq / nyq, Q=30)

    return (bp_b, bp_a), (notch_b, notch_a)


def apply_filters(signal_array, bp_coeffs, notch_coeffs):
    """Apply bandpass then notch filter using zero-phase filtfilt.
    
    signal_array: shape (N, C) where N=samples, C=channels
    Returns filtered array of same shape.
    """
    bp_b, bp_a = bp_coeffs
    notch_b, notch_a = notch_coeffs
    filtered = np.zeros_like(signal_array, dtype=np.float32)
    for c in range(signal_array.shape[1]):
        x = signal_array[:, c].astype(np.float64)
        x = scipy_signal.filtfilt(bp_b, bp_a, x)
        x = scipy_signal.filtfilt(notch_b, notch_a, x)
        filtered[:, c] = x.astype(np.float32)
    return filtered


bp_coeffs, notch_coeffs = build_filters(
    FS, BANDPASS_LOW, BANDPASS_HIGH, NOTCH_FREQ, FILTER_ORDER
)
print('Filters built successfully.')

In [ ]:
# ─── Preprocess entire dataset ────────────────────────────────────────────────
print('Filtering signals...')
raw_signals = raw_df[signal_cols].values.astype(np.float32)
labels_raw  = raw_df[LABEL_COL].values

filtered_signals = apply_filters(raw_signals, bp_coeffs, notch_coeffs)

print(f'Filtered signals shape: {filtered_signals.shape}')

# ─── Compare raw vs filtered for one channel ─────────────────────────────────
seg = slice(0, PREVIEW_SAMPLES)
t   = np.arange(PREVIEW_SAMPLES) / FS

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
axes[0].plot(t, raw_signals[seg, 0], lw=0.5, color='gray', label='Raw')
axes[0].set_title('CH1 — Raw', fontweight='bold'); axes[0].legend()
axes[1].plot(t, filtered_signals[seg, 0], lw=0.5, color='steelblue', label='Filtered')
axes[1].set_title('CH1 — Filtered (20–450 Hz bandpass + 60 Hz notch)', fontweight='bold')
axes[1].legend(); axes[1].set_xlabel('Time (s)')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('filter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Windowing — Segment into Fixed-Length Epochs

In [ ]:
def sliding_window_dataset(signals, labels, window_len, step_len, purity_thresh=0.80):
    """
    Segment a continuous EMG recording into overlapping windows.

    Majority-vote labeling: a window is kept only if ≥ purity_thresh
    of its samples share the same label (avoids training on transition noise).

    Parameters
    ----------
    signals      : (N, C) float32 — filtered EMG
    labels       : (N,) str       — per-sample label
    window_len   : int            — samples per window
    step_len     : int            — stride in samples
    purity_thresh: float          — minimum label agreement fraction

    Returns
    -------
    windows      : (M, C, W) float32 — channel-first for PyTorch
    window_labels: (M,) str
    """
    windows, window_labels = [], []
    N = len(signals)

    for start in range(0, N - window_len + 1, step_len):
        end     = start + window_len
        seg_sig = signals[start:end]          # (W, C)
        seg_lbl = labels[start:end]

        # Majority vote on label
        counts      = Counter(seg_lbl)
        majority_lbl, majority_cnt = counts.most_common(1)[0]
        purity      = majority_cnt / window_len

        if purity >= purity_thresh:
            windows.append(seg_sig.T)         # (C, W) — channel-first
            window_labels.append(majority_lbl)

    return np.array(windows, dtype=np.float32), np.array(window_labels)


print('Windowing data...')
windows, window_labels = sliding_window_dataset(
    filtered_signals, labels_raw,
    WINDOW_LEN, STEP_LEN, LABEL_PURITY_THRESHOLD
)

print(f'Windows shape : {windows.shape}   (samples × channels × time)')
print(f'Label shape   : {window_labels.shape}')

window_label_counts = Counter(window_labels)
print(f'\nWindow class distribution:')
for k, v in sorted(window_label_counts.items()):
    print(f'  {k:30s}: {v:>5,} windows')

## 5. Feature Engineering (for MLP baseline)

In [ ]:
def extract_features(window):
    """
    Extract a rich set of time-domain and frequency-domain features
    from a single EMG window.

    Parameters
    ----------
    window : (C, W) float32 — C channels, W time samples

    Returns
    -------
    features : 1D float32 array of length (C × 12)
    """
    C, W = window.shape
    feats = []
    freqs = np.fft.rfftfreq(W, d=1.0/FS)

    for c in range(C):
        x = window[c]

        # ── Time domain ───────────────────────────────────────────────────────
        mav   = np.mean(np.abs(x))                          # Mean Absolute Value
        rms   = np.sqrt(np.mean(x**2))                      # Root Mean Square
        var   = np.var(x)                                    # Variance
        std   = np.std(x)                                    # Std dev
        wl    = np.sum(np.abs(np.diff(x)))                  # Waveform Length
        zc    = np.sum(np.diff(np.sign(x)) != 0)            # Zero Crossings
        ssc   = np.sum(                                       # Slope Sign Changes
            np.diff(np.sign(np.diff(x))) != 0
        )
        sk    = float(skew(x))                               # Skewness
        kurt  = float(kurtosis(x))                           # Kurtosis

        # ── Frequency domain ──────────────────────────────────────────────────
        mag   = np.abs(np.fft.rfft(x))
        psd   = mag**2
        total_power = np.sum(psd) + 1e-10
        mnf   = np.sum(freqs * psd) / total_power           # Mean frequency
        mdf   = freqs[np.searchsorted(                       # Median frequency
                    np.cumsum(psd), total_power / 2
                )]
        pkf   = freqs[np.argmax(psd)]                        # Peak frequency

        feats.extend([mav, rms, var, std, wl, zc, ssc, sk, kurt, mnf, mdf, pkf])

    return np.array(feats, dtype=np.float32)


print('Extracting handcrafted features...')
features = np.array(
    [extract_features(w) for w in tqdm(windows, desc='Features')],
    dtype=np.float32
)
print(f'Feature matrix shape: {features.shape}  (samples × features)')
print(f'Features per channel: {features.shape[1] // N_CHANNELS}')
print(f'Total features: {features.shape[1]}')

## 6. Label Encoding & Train / Val / Test Split

In [ ]:
# Encode string labels → integers
le = LabelEncoder()
y_encoded = le.fit_transform(window_labels)
N_CLASSES  = len(le.classes_)

print(f'Number of gesture classes: {N_CLASSES}')
print('Class mapping:')
for i, name in enumerate(le.classes_):
    print(f'  {i:2d}  →  {name}')

# ─── Stratified split: 70% train / 15% val / 15% test ─────────────────────
X_raw = windows    # (M, C, W) — raw for CNN
X_feat = features  # (M, F)   — features for MLP
y = y_encoded

# First split off test set
idx = np.arange(len(y))
idx_trainval, idx_test = train_test_split(
    idx, test_size=TEST_SPLIT, stratify=y, random_state=SEED
)
# Then split train into train+val
val_fraction_of_trainval = VAL_SPLIT / (1 - TEST_SPLIT)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=val_fraction_of_trainval,
    stratify=y[idx_trainval], random_state=SEED
)

print(f'\nSplit sizes — Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}')

# Normalise features (MLP path only — StandardScaler on train, apply to val/test)
scaler = StandardScaler()
X_feat_train = scaler.fit_transform(X_feat[idx_train])
X_feat_val   = scaler.transform(X_feat[idx_val])
X_feat_test  = scaler.transform(X_feat[idx_test])

# CNN inputs: per-window z-score normalisation (per-channel, preserves shape)
def norm_windows(win_arr):
    """Normalise each window per channel: (M, C, W) -> (M, C, W)"""
    mu  = win_arr.mean(axis=2, keepdims=True)
    std = win_arr.std(axis=2, keepdims=True) + 1e-8
    return (win_arr - mu) / std

X_raw_train = norm_windows(X_raw[idx_train])
X_raw_val   = norm_windows(X_raw[idx_val])
X_raw_test  = norm_windows(X_raw[idx_test])

y_train = y[idx_train]
y_val   = y[idx_val]
y_test  = y[idx_test]

## 7. PyTorch Datasets & DataLoaders

In [ ]:
class EMGDataset(Dataset):
    """Generic EMG dataset that holds either raw windows or feature vectors."""
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x, label = self.X[idx], self.y[idx]

        # ── Simple augmentation (training only) ───────────────────────────────
        if self.augment and x.dim() == 2:  # (C, W) windows only
            # Gaussian noise injection
            if torch.rand(1) < 0.4:
                x = x + torch.randn_like(x) * 0.05
            # Random amplitude scaling ±20%
            if torch.rand(1) < 0.4:
                x = x * (0.8 + torch.rand(1) * 0.4)
            # Random sign flip (simulates electrode orientation variance)
            if torch.rand(1) < 0.2:
                x = -x

        return x, label


def make_weighted_sampler(y):
    """Inverse-frequency weighted sampler to handle class imbalance."""
    class_counts = np.bincount(y)
    weights      = 1.0 / class_counts[y]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True
    )


# CNN datasets (raw windows: shape C × W)
cnn_train_ds = EMGDataset(X_raw_train, y_train, augment=True)
cnn_val_ds   = EMGDataset(X_raw_val,   y_val,   augment=False)
cnn_test_ds  = EMGDataset(X_raw_test,  y_test,  augment=False)

# MLP datasets (feature vectors)
mlp_train_ds = EMGDataset(X_feat_train, y_train, augment=False)
mlp_val_ds   = EMGDataset(X_feat_val,   y_val,   augment=False)
mlp_test_ds  = EMGDataset(X_feat_test,  y_test,  augment=False)

cnn_train_loader = DataLoader(
    cnn_train_ds, batch_size=BATCH_SIZE,
    sampler=make_weighted_sampler(y_train), num_workers=0
)
cnn_val_loader   = DataLoader(cnn_val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
cnn_test_loader  = DataLoader(cnn_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

mlp_train_loader = DataLoader(
    mlp_train_ds, batch_size=BATCH_SIZE,
    sampler=make_weighted_sampler(y_train), num_workers=0
)
mlp_val_loader   = DataLoader(mlp_val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
mlp_test_loader  = DataLoader(mlp_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('DataLoaders ready.')
print(f'  CNN input shape per batch: (batch, {N_CHANNELS}, {WINDOW_LEN})')
print(f'  MLP input dim: {X_feat_train.shape[1]}')

## 8. Model Architectures

In [ ]:
# ─── Model A: 1D-CNN (operates directly on raw filtered windows) ──────────────
#
# Architecture: 3 × (Conv1D → BatchNorm → ReLU → MaxPool → Dropout)
#               → Global Average Pooling → FC head
#
# Why CNN for EMG?
#   - Learns spatial (cross-channel) and temporal patterns jointly
#   - No manual feature engineering needed
#   - Proven on NinaPro and similar multi-class EMG datasets

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride=1, pool=2, dropout=0.25):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.block(x)


class EMG_CNN(nn.Module):
    """
    1D Convolutional Network for EMG gesture classification.

    Input : (batch, n_channels, window_len)
    Output: (batch, n_classes) — raw logits
    """
    def __init__(self, n_channels, window_len, n_classes, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(n_channels, 32,  kernel=7,  pool=2, dropout=0.2),  # temporal: coarse
            ConvBlock(32,         64,  kernel=5,  pool=2, dropout=0.25), # temporal: mid
            ConvBlock(64,         128, kernel=3,  pool=2, dropout=0.25), # temporal: fine
            ConvBlock(128,        256, kernel=3,  pool=2, dropout=0.3),  # deep patterns
        )
        # Global average pool collapses remaining time dim → (batch, 256)
        self.gap = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = self.features(x)   # (batch, 256, T')
        x = self.gap(x)        # (batch, 256, 1)
        x = self.classifier(x) # (batch, n_classes)
        return x


# ─── Model B: MLP on handcrafted features ────────────────────────────────────
class EMG_MLP(nn.Module):
    """
    Fully-connected baseline using handcrafted EMG features.

    Input : (batch, n_features)
    Output: (batch, n_classes) — raw logits
    """
    def __init__(self, n_features, n_classes, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout/2),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)


# Instantiate both models
cnn_model = EMG_CNN(N_CHANNELS, WINDOW_LEN, N_CLASSES).to(DEVICE)
mlp_model = EMG_MLP(X_feat_train.shape[1], N_CLASSES).to(DEVICE)

# Parameter counts
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'CNN parameters: {count_params(cnn_model):,}')
print(f'MLP parameters: {count_params(mlp_model):,}')
print('\nCNN architecture:')
print(cnn_model)

## 9. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        # Gradient clipping — prevents exploding gradients on noisy EMG data
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        preds  = logits.argmax(1)
        total_loss += loss.item() * len(y_batch)
        correct    += (preds == y_batch).sum().item()
        total      += len(y_batch)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_targets)


def train_model(model, train_loader, val_loader, name='Model', n_epochs=N_EPOCHS):
    """
    Full training loop with:
    - Label-smoothed cross-entropy (reduces overconfidence on noisy EMG labels)
    - AdamW optimizer (weight decay for regularization)
    - Cosine annealing LR schedule
    - Best-model checkpointing by validation accuracy
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)

    history   = {'train_loss': [], 'val_loss': [],
                 'train_acc':  [], 'val_acc': []}
    best_val_acc  = 0.0
    best_state    = None
    patience      = 15
    no_improve    = 0

    print(f'\n{"═"*55}')
    print(f'  Training: {name}')
    print(f'{"═"*55}')
    print(f'{"Epoch":>6}  {"Train Loss":>10}  {"Train Acc":>10}  {"Val Loss":>9}  {"Val Acc":>8}  {"LR":>8}')
    print(f'{"─"*65}')

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        lr_now = scheduler.get_last_lr()[0]

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve   = 0
            marker       = '  ✓ BEST'
        else:
            no_improve  += 1
            marker       = ''

        if epoch % 5 == 0 or epoch == 1:
            print(f'{epoch:>6}  {tr_loss:>10.4f}  {tr_acc*100:>9.2f}%'
                  f'  {vl_loss:>9.4f}  {vl_acc*100:>7.2f}%  {lr_now:>8.6f}{marker}')

        if no_improve >= patience:
            print(f'  Early stopping at epoch {epoch} (no val improvement for {patience} epochs)')
            break

    # Restore best weights
    model.load_state_dict(best_state)
    print(f'\n  Best val accuracy: {best_val_acc*100:.2f}%')
    return history, best_val_acc


print('Training functions defined.')

In [ ]:
# ─── Train CNN ────────────────────────────────────────────────────────────────
cnn_history, cnn_best_val = train_model(
    cnn_model, cnn_train_loader, cnn_val_loader, name='1D-CNN'
)

In [ ]:
# ─── Train MLP ────────────────────────────────────────────────────────────────
mlp_history, mlp_best_val = train_model(
    mlp_model, mlp_train_loader, mlp_val_loader, name='MLP (Handcrafted Features)'
)

## 10. Training Curves

In [ ]:
def plot_training_curves(history, name, ax_loss, ax_acc):
    epochs = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='Train', linewidth=2)
    ax_loss.plot(epochs, history['val_loss'],   label='Val',   linewidth=2, linestyle='--')
    ax_loss.set_title(f'{name} — Loss', fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Cross-Entropy Loss')
    ax_loss.legend(); ax_loss.grid(True, alpha=0.3)

    ax_acc.plot(epochs, [v*100 for v in history['train_acc']], label='Train', linewidth=2)
    ax_acc.plot(epochs, [v*100 for v in history['val_acc']],   label='Val',   linewidth=2, linestyle='--')
    ax_acc.set_title(f'{name} — Accuracy', fontweight='bold')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy (%)')
    ax_acc.legend(); ax_acc.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(15, 9))
plot_training_curves(cnn_history, '1D-CNN',             axes[0][0], axes[0][1])
plot_training_curves(mlp_history, 'MLP (Feat.)',        axes[1][0], axes[1][1])
plt.suptitle('Training Curves', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Test Set Evaluation

In [ ]:
criterion = nn.CrossEntropyLoss()

_, cnn_test_acc, cnn_preds, cnn_targets = evaluate(cnn_model, cnn_test_loader, criterion)
_, mlp_test_acc, mlp_preds, mlp_targets = evaluate(mlp_model, mlp_test_loader, criterion)

print('═'*55)
print(f'  CNN Test Accuracy : {cnn_test_acc*100:.2f}%')
print(f'  MLP Test Accuracy : {mlp_test_acc*100:.2f}%')
print('═'*55)

# Classification report
print('\n── CNN Full Report ──')
print(classification_report(cnn_targets, cnn_preds, target_names=le.classes_))

print('\n── MLP Full Report ──')
print(classification_report(mlp_targets, mlp_preds, target_names=le.classes_))

In [ ]:
def plot_confusion_matrix(targets, preds, class_names, title, ax):
    cm = confusion_matrix(targets, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalize

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5, vmin=0, vmax=1
    )
    ax.set_title(f'{title}\nRow-Normalized Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted Gesture')
    ax.set_ylabel('True Gesture')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)


fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_confusion_matrix(cnn_targets, cnn_preds, le.classes_, f'1D-CNN  (Acc: {cnn_test_acc*100:.1f}%)', axes[0])
plot_confusion_matrix(mlp_targets, mlp_preds, le.classes_, f'MLP     (Acc: {mlp_test_acc*100:.1f}%)', axes[1])
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Per-class F1 comparison bar chart ───────────────────────────────────────
cnn_f1 = f1_score(cnn_targets, cnn_preds, average=None)
mlp_f1 = f1_score(mlp_targets, mlp_preds, average=None)

x     = np.arange(N_CLASSES)
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - width/2, cnn_f1, width, label='CNN',  color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, mlp_f1, width, label='MLP',  color='coral',     alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(le.classes_, rotation=45, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score: CNN vs MLP', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.axhline(0.75, color='gray', linestyle=':', linewidth=1.2, label='75% target')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Error Analysis

In [ ]:
# Which gesture pairs are most commonly confused? (CNN model)
cm_cnn = confusion_matrix(cnn_targets, cnn_preds)

# Extract off-diagonal errors
errors = []
for true_idx in range(N_CLASSES):
    for pred_idx in range(N_CLASSES):
        if true_idx != pred_idx and cm_cnn[true_idx, pred_idx] > 0:
            errors.append({
                'True':       le.classes_[true_idx],
                'Predicted':  le.classes_[pred_idx],
                'Count':      cm_cnn[true_idx, pred_idx],
                'Error Rate': cm_cnn[true_idx, pred_idx] / cm_cnn[true_idx].sum()
            })

error_df = pd.DataFrame(errors).sort_values('Count', ascending=False)
print('Top CNN Confusions:')
print(error_df.head(15).to_string(index=False))

## 13. Model Export

In [ ]:
# ─── Save full model checkpoints ─────────────────────────────────────────────
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'n_channels':  N_CHANNELS,
    'window_len':  WINDOW_LEN,
    'n_classes':   N_CLASSES,
    'label_encoder_classes': le.classes_.tolist(),
    'fs':          FS,
    'val_acc':     cnn_best_val,
    'test_acc':    cnn_test_acc,
}, 'emg_cnn_best.pth')

torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'scaler_mean':  scaler.mean_,
    'scaler_scale': scaler.scale_,
    'n_features':   X_feat_train.shape[1],
    'n_classes':    N_CLASSES,
    'label_encoder_classes': le.classes_.tolist(),
    'val_acc':      mlp_best_val,
    'test_acc':     mlp_test_acc,
}, 'emg_mlp_best.pth')

print('Checkpoints saved: emg_cnn_best.pth | emg_mlp_best.pth')

# ─── TorchScript export for Raspberry Pi deployment ───────────────────────────
cnn_model.eval()
dummy_input = torch.randn(1, N_CHANNELS, WINDOW_LEN)
scripted    = torch.jit.trace(cnn_model, dummy_input)
scripted.save('emg_cnn_scripted.pt')
print('TorchScript model saved: emg_cnn_scripted.pt  (ready for Raspberry Pi)')
print('  → Load on Pi with: model = torch.jit.load("emg_cnn_scripted.pt")')

## 14. Real-Time Inference Helper

Drop-in function for the integration team to call from the Pi-side control loop.

In [ ]:
class RealTimeClassifier:
    """
    Stateful real-time EMG gesture classifier.

    Maintains an internal ring buffer of the last WINDOW_LEN samples.
    Call .update(new_samples) each time a new batch of samples arrives.
    Emits a prediction every STEP_LEN new samples (matching training stride).

    Example usage (Raspberry Pi integration):
    ─────────────────────────────────────────
        classifier = RealTimeClassifier('emg_cnn_scripted.pt', label_classes)
        while True:
            new_data = read_from_esp32()   # shape (n_new, 3)
            gesture  = classifier.update(new_data)
            if gesture:
                motor_controller.set_gesture(gesture)
    """
    def __init__(self, model_path, label_classes,
                 fs=FS, window_len=WINDOW_LEN, step_len=STEP_LEN,
                 bp_low=BANDPASS_LOW, bp_high=BANDPASS_HIGH, notch=NOTCH_FREQ):
        self.model        = torch.jit.load(model_path)
        self.model.eval()
        self.label_classes = label_classes
        self.window_len    = window_len
        self.step_len      = step_len
        self.buffer        = np.zeros((window_len, 3), dtype=np.float32)
        self.samples_since_last_pred = 0

        bp_coeffs, notch_coeffs = build_filters(fs, bp_low, bp_high, notch)
        self._bp_coeffs    = bp_coeffs
        self._notch_coeffs = notch_coeffs

    def update(self, new_samples):
        """
        new_samples : (N, 3) numpy float32 — new raw EMG rows
        Returns predicted gesture string, or None if not yet ready.
        """
        n = len(new_samples)
        # Shift buffer and insert new samples
        self.buffer = np.roll(self.buffer, -n, axis=0)
        self.buffer[-n:] = new_samples
        self.samples_since_last_pred += n

        if self.samples_since_last_pred < self.step_len:
            return None
        self.samples_since_last_pred = 0

        # Filter the window
        filtered = apply_filters(self.buffer, self._bp_coeffs, self._notch_coeffs)
        # Normalise (C, W) and build batch
        win = filtered.T.astype(np.float32)         # (C, W)
        mu  = win.mean(axis=1, keepdims=True)
        std = win.std(axis=1, keepdims=True) + 1e-8
        win = (win - mu) / std
        tensor = torch.tensor(win).unsqueeze(0)     # (1, C, W)

        with torch.no_grad():
            logits = self.model(tensor)
            pred   = logits.argmax(1).item()

        return self.label_classes[pred]


# Quick smoke-test
rt = RealTimeClassifier('emg_cnn_scripted.pt', le.classes_.tolist())
dummy_chunk = np.random.randn(STEP_LEN, 3).astype(np.float32)

# First call: buffer not full yet → None
result = rt.update(dummy_chunk)
print(f'First chunk result (buffer filling): {result}')

# Feed enough chunks to fill buffer
for _ in range((WINDOW_LEN // STEP_LEN)):
    result = rt.update(dummy_chunk)
print(f'After buffer fill — predicted gesture: {result}')
print('\nRealTimeClassifier smoke test passed ✓')

## 15. Summary

In [ ]:
print('═'*60)
print('  EMG GESTURE CLASSIFICATION — FINAL SUMMARY')
print('═'*60)
print(f'  Dataset')
print(f'    CSV files loaded    : {len(csv_paths)}')
print(f'    Total raw samples   : {len(raw_df):,}')
print(f'    Gesture classes     : {N_CLASSES}  →  {list(le.classes_)}')
print()
print(f'  Windowing')
print(f'    Window size         : {WINDOW_LEN} samples ({WINDOW_MS} ms)')
print(f'    Overlap             : {int(OVERLAP_PCT*100)}%')
print(f'    Windows created     : {len(windows):,}')
print()
print(f'  Results')
print(f'    CNN  test accuracy  : {cnn_test_acc*100:.2f}%  (val: {cnn_best_val*100:.2f}%)')
print(f'    MLP  test accuracy  : {mlp_test_acc*100:.2f}%  (val: {mlp_best_val*100:.2f}%)')
print()
print(f'  Best model: {"CNN" if cnn_test_acc >= mlp_test_acc else "MLP"}')
print()
print(f'  Saved artifacts')
print(f'    emg_cnn_best.pth         — CNN checkpoint')
print(f'    emg_mlp_best.pth         — MLP checkpoint')
print(f'    emg_cnn_scripted.pt      — TorchScript for Raspberry Pi')
print(f'    class_distribution.png   — Class balance plot')
print(f'    raw_signal_preview.png   — Raw signal overview')
print(f'    filter_comparison.png    — Raw vs filtered')
print(f'    training_curves.png      — Loss & accuracy history')
print(f'    confusion_matrices.png   — Normalized confusion matrices')
print(f'    per_class_f1.png         — Per-gesture F1 comparison')
print('═'*60)